<a href="https://colab.research.google.com/github/ek182238/CSC349/blob/main/HW2/data_prep_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
# Task 1
# Load the dataset into a Pandas DataFrame
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('cybersecurity_intrusion_data.csv')

In [ ]:
# Task 2
# Use structural inspection methods to analyze features
df.head()

,session_id,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
0,SID_00001,599,TCP,4,492.983263,DES,0.606818,1,Edge,0,1
1,SID_00002,472,TCP,3,1557.996461,DES,0.301569,0,Firefox,0,0
2,SID_00003,629,TCP,3,75.044262,DES,0.739164,2,Chrome,0,1
3,SID_00004,804,UDP,4,601.248835,DES,0.123267,0,Unknown,0,1
4,SID_00005,453,TCP,5,532.540888,AES,0.054874,1,Firefox,0,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9537 entries, 0 to 9536
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   session_id           9537 non-null   object 
 1   network_packet_size  9537 non-null   int64  
 2   protocol_type        9537 non-null   object 
 3   login_attempts       9537 non-null   int64  
 4   session_duration     9537 non-null   float64
 5   encryption_used      9537 non-null   object 
 6   ip_reputation_score  9537 non-null   float64
 7   failed_logins        9537 non-null   int64  
 8   browser_type         9537 non-null   object 
 9   unusual_time_access  9537 non-null   int64  
 10  attack_detected      9537 non-null   int64  
dtypes: float64(2), int64(5), object(4)
memory usage: 819.7+ KB


In [ ]:
df['network_packet_size'].describe()

,network_packet_size
count,9537.000000
mean,500.430639
std,198.379364
min,64.000000
25%,365.000000
50%,499.000000
75%,635.000000
max,1285.000000


In [ ]:
df['session_duration'].describe()

,session_duration
count,9537.000000
mean,792.745312
std,786.560144
min,0.500000
25%,231.953006
50%,556.277457
75%,1105.380602
max,7190.392213


In [ ]:
df['login_attempts'].describe()

,login_attempts
count,9537.000000
mean,4.032086
std,1.963012
min,1.000000
25%,3.000000
50%,4.000000
75%,5.000000
max,13.000000


In [ ]:
# Task 3
# Drop any non-informative identifiers
df.drop(columns='session_id')

,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
0,599,TCP,4,492.983263,DES,0.606818,1,Edge,0,1
1,472,TCP,3,1557.996461,DES,0.301569,0,Firefox,0,0
2,629,TCP,3,75.044262,DES,0.739164,2,Chrome,0,1
3,804,UDP,4,601.248835,DES,0.123267,0,Unknown,0,1
4,453,TCP,5,532.540888,AES,0.054874,1,Firefox,0,0
...,...,...,...,...,...,...,...,...,...,...
9532,194,ICMP,3,226.049889,AES,0.517737,3,Chrome,0,1
9533,380,TCP,3,182.848475,AES,0.408485,0,Chrome,0,0
9534,664,TCP,5,35.170248,AES,0.359200,1,Firefox,0,0
9535,406,TCP,4,86.664703,AES,0.537417,1,Chrome,1,0


In [ ]:
# Task 4
# Quantify any missing values across columns and implement a structured imputation strategy to handle missing entries
print('Missing values count:\n')
print(df.isnull().sum())

Missing values count:

session_id             0
network_packet_size    0
protocol_type          0
login_attempts         0
session_duration       0
encryption_used        0
ip_reputation_score    0
failed_logins          0
browser_type           0
unusual_time_access    0
attack_detected        0
dtype: int64


In [ ]:
# Imputation using Pandas. I used the mode because the data is categorical
df['encryption_used'] = df['encryption_used'].fillna(df['encryption_used'].mode()[0])

In [ ]:
print('Missing values after imputation:\n')
print(df.isnull().sum())

Missing values after imputation:

session_id             0
network_packet_size    0
protocol_type          0
login_attempts         0
session_duration       0
encryption_used        0
ip_reputation_score    0
failed_logins          0
browser_type           0
unusual_time_access    0
attack_detected        0
dtype: int64


In [34]:
# Task 5
# Isolate numerical and categorical features, then build a Scikit-Learn ColumnTransformer and Pipeline that
# applies appropriate scaling (StandardScaler) to numerical columns and encoding (OneHotEncoder) to categorical columns

# I first separate the data into a DataFrame (X) containing everything except the target, and a series (y) with the target
X = df.drop(columns='attack_detected')
y = df['attack_detected']

# Data is grouped into numerical & categorical
numerical_data = X.select_dtypes(include=['int64', 'float64'])
categorical_data = X.select_dtypes(include=['object'])

print('Numerical:', numerical_data)
print('Categorical:', categorical_data)

Numerical:       network_packet_size  login_attempts  session_duration  \
0                     599               4        492.983263   
1                     472               3       1557.996461   
2                     629               3         75.044262   
3                     804               4        601.248835   
4                     453               5        532.540888   
...                   ...             ...               ...   
9532                  194               3        226.049889   
9533                  380               3        182.848475   
9534                  664               5         35.170248   
9535                  406               4         86.664703   
9536                  340               6         86.876744   

      ip_reputation_score  failed_logins  unusual_time_access  
0                0.606818              1                    0  
1                0.301569              0                    0  
2                0.739164              2

In [35]:
# ColumnTransformer (using StandardScaler and OneHotEncoder) and Pipeline

column_transformer = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_data.columns),
        ('cat', OneHotEncoder(), categorical_data.columns)
    ])

pipeline = Pipeline(steps=[('preprocessor', column_transformer)])

In [37]:
X_transformed = pipeline.fit_transform(X)
print('Shape:', X_transformed.shape)
print(X_transformed)

Shape: (9537, 9554)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 95370 stored elements and shape (9537, 9554)>
  Coords	Values
  (0, 0)	0.4968991118754836
  (0, 1)	-0.016345922119364028
  (0, 2)	-0.3811250382488641
  (0, 3)	1.5549300182169943
  (0, 4)	-0.500779475267592
  (0, 5)	-0.41998901581553155
  (0, 6)	1.0
  (0, 9544)	1.0
  (0, 9547)	1.0
  (0, 9550)	1.0
  (1, 0)	-0.14332201048794352
  (1, 1)	-0.5257938281728741
  (1, 2)	0.9729596374141649
  (1, 3)	-0.16802943390254468
  (1, 4)	-1.4679592759211006
  (1, 5)	-0.41998901581553155
  (1, 7)	1.0
  (1, 9544)	1.0
  (1, 9547)	1.0
  (1, 9551)	1.0
  (2, 0)	0.6481324478668443
  (2, 1)	-0.5257938281728741
  (2, 2)	-0.912503239856203
  (2, 3)	2.3019501029733784
  (2, 4)	0.4664003253859165
  :	:
  (9534, 5)	-0.41998901581553155
  (9534, 9540)	1.0
  (9534, 9544)	1.0
  (9534, 9546)	1.0
  (9534, 9551)	1.0
  (9535, 0)	-0.4760353496689372
  (9535, 1)	-0.016345922119364028
  (9535, 2)	-0.8977287178625879
  (9535, 3)	1.1631984252579846

In [ ]:
# Task 6
# Compute the Pearson correlation matrix for the numerical features, generate a Seaborn heatmap with annotations,
# and create at least two distribution plots and one scatter plot to examine feature relationships and the target variable